In [5]:
import pandas as pd
import statsmodels.api as sm

# 1. Load the datasets
print("Loading data...")
signals_df = pd.read_csv('../examples/monthly_signals_decay.csv')
green_df = pd.read_csv('../green cleaned.csv')

# 2. Standardize merging keys (Date and Ticker)
print("Formatting dates and tickers...")
# Ensure 'year_month' is a string for merging
signals_df['year_month'] = signals_df['year_month'].astype(str)
signals_df['ticker'] = signals_df['symbol'].str.upper()

# Convert 'datadate' to datetime, then extract 'YYYY-MM' to match signals_df
green_df['datadate'] = pd.to_datetime(green_df['datadate'])
green_df['year_month'] = green_df['datadate'].dt.strftime('%Y-%m')
green_df['ticker'] = green_df['ticker'].str.upper()

# 3. Merge the datasets
print("Merging datasets...")
merged_df = pd.merge(signals_df, green_df, on=['ticker', 'year_month'], how='inner')

Loading data...


/var/folders/w3/6myr5tjs6ls3zn1g3ywbc29r0000gn/T/ipykernel_72771/1249238544.py:7: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  green_df = pd.read_csv('../green cleaned.csv')


Formatting dates and tickers...
Merging datasets...


In [16]:
# 4. Prepare data for regression
# Drop rows missing the variables we need for the regression
# regression_vars = ['weighted_sentiment', 'mve', 'bm', 'mom12m']

# variables = ['agr', 'bm', 'mom12m', 'mve', 'operprof', 'roeq', 'absacc',
#             'acc', 'aeavol', 'age', 'baspread', 'BETA', 'bm_ia', 'cash',
#             'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho',
#             'chempia', 'chfeps', 'chinv', 'chmom', 'chnanalyst', 'chpmia',
#             'chtx', 'cinvest', 'convind', 'currat', 'depr', 'disp', 'divi',
#             'divo', 'dy', 'ear', 'egr', 'ep', 'fgr5yr', 'gma', 'grcapx',
#             'grltnoa', 'herf', 'hire', 'idiovol', 'ill', 'indmom', 'invest',
#             'IPO', 'lev', 'mom1m', 'mom36m', 'ms', 'mve_ia', 'nanalyst',
#             'nincr', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr',
#             'pchgm_pchsale', 'pchsale_pchinvt', 'pchsale_pchrect',
#             'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'pricedelay', 'ps',
#             'rd', 'rd_mve', 'rd_sale', 'realestate', 'retvol', 'roaq',
#             'roavol', 'roic', 'rsup', 'salecash', 'saleinv', 'salerec',
#             'secured', 'securedind', 'sfe', 'sgr', 'sin', 'sp', 'std_dolvol',
#             'std_turn', 'stdcf', 'sue', 'tang', 'tb', 'turn', 'zerotrade']

variables = ['mve', 'bm', 'mom12m'] 

reg_df = merged_df.dropna(subset='weighted_sentiment_score').copy()

# Define dependent variable (Y) and independent variables (X)
Y = reg_df['weighted_sentiment_score']
X = reg_df[variables]

# Add a constant (intercept) to the model
X = sm.add_constant(X)

# 5. Run the Regression
print("Running regression...")
model = sm.OLS(Y, X).fit()

# Print the summary to see how much of the sentiment is explained by characteristics
print("\nRegression Summary:")
print(model.summary())

# 6. Extract the residuals
# The residuals represent the sentiment NOT captured by mve, bm, and mom12m
reg_df['residual_sentiment'] = model.resid

print("\nSample of isolated news sentiment (residuals):")
print(reg_df[['ticker', 'year_month', 'weighted_sentiment_score', 'residual_sentiment']].head())

# 7. Save the results
# reg_df.to_csv('orthogonalized_sentiment.csv', index=False)
# print("\nSaved residuals to 'orthogonalized_sentiment.csv'")

Running regression...

Regression Summary:
                               OLS Regression Results                               
Dep. Variable:     weighted_sentiment_score   R-squared:                       0.000
Model:                                  OLS   Adj. R-squared:                  0.000
Method:                       Least Squares   F-statistic:                     3.682
Date:                      Wed, 26 Aug 2026   Prob (F-statistic):             0.0115
Time:                              20:28:05   Log-Likelihood:                 21325.
No. Observations:                     22892   AIC:                        -4.264e+04
Df Residuals:                         22888   BIC:                        -4.261e+04
Df Model:                                 3                                         
Covariance Type:                  nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------

In [15]:
import numpy as np

print("Mapping continuous residuals to buy/sell/hold classifications using absolute thresholds...")

# 1. Apply your exact fixed thresholds to the residuals
conditions = [
    reg_df['residual_sentiment'] > 0.1,
    reg_df['residual_sentiment'] < -0.1
]
choices = ['buy', 'sell']
# This overwrites the old 'signal' column that came from the original FinBERT CSV
reg_df['signal'] = np.select(conditions, choices, default='hold')

# 2. Drop the old sentiment score columns to prevent duplicates
columns_to_drop = ['avg_sentiment_score', 'weighted_sentiment_score']
existing_cols_to_drop = [c for c in columns_to_drop if c in reg_df.columns]
reg_df = reg_df.drop(columns=existing_cols_to_drop)

# 3. Format columns to perfectly match your downstream FinBERT load function
output_df = reg_df.rename(columns={
    'ticker': 'symbol', 
    'residual_sentiment': 'avg_sentiment_score' 
}).copy()

# Ensure 'company' exists for the load function (fill with symbol if missing)
if 'company' not in output_df.columns:
    output_df['company'] = output_df['symbol']

# 4. Save the formatted output
output_cols = ['symbol', 'company', 'year_month', 'signal', 'avg_sentiment_score']
output_df[output_cols].to_csv('orthogonalized_sentiment_96.csv', index=False)

print("\nSaved fully formatted signals to 'orthogonalized_sentiment.csv'")
print("\nSignal Distribution based on +/- 0.1 thresholds:")
print(output_df['signal'].value_counts())

Mapping continuous residuals to buy/sell/hold classifications using absolute thresholds...

Saved fully formatted signals to 'orthogonalized_sentiment.csv'

Signal Distribution based on +/- 0.1 thresholds:
signal
hold    21997
sell      494
buy       401
Name: count, dtype: int64
